# Notebook 04 — Gradio Deployment
**YOLOv7 Gym Equipment Detector**  
**Team:** Varun Gazala · Mohit Raiyani · Jatinkumar Nabhoya · UNH Spring 2026

Loads the fine-tuned `best.pt` checkpoint and launches a Gradio web app.  
Upload any gym image → side-by-side **Original | Detected** output.

**Run all cells top-to-bottom. The last cell opens the app in your browser.**

In [10]:
# ── Step 0: Environment Setup ─────────────────────────────────────────────────
import os, sys, subprocess

YOLOV7_DIR   = os.path.abspath('../yolov7')
WEIGHTS_PATH = os.path.abspath('../best.pt')
COCO_WEIGHTS = os.path.join(YOLOV7_DIR, 'yolov7.pt')

# Clone YOLOv7 framework if not present
if not os.path.exists(YOLOV7_DIR):
    print('Cloning YOLOv7 ...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/WongKinYiu/yolov7.git', YOLOV7_DIR],
        check=True
    )

# Download COCO pretrained weights if not present
# (needed to reconstruct the model architecture even though we override the weights)
if not os.path.exists(COCO_WEIGHTS):
    print('Downloading yolov7.pt (~72 MB) ...')
    subprocess.run(
        ['curl', '-L', '-o', COCO_WEIGHTS,
         'https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt'],
        check=True
    )

# Install Gradio if missing
try:
    import gradio as gr
    print(f'Gradio {gr.__version__} already installed.')
except ImportError:
    print('Installing Gradio ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True)

# Guard: make sure best.pt is actually present
assert os.path.exists(WEIGHTS_PATH), (
    f'\nbest.pt not found at: {WEIGHTS_PATH}\n'
    'Run 02_phase2_finetuning.ipynb first, or copy best.pt to the project root folder.'
)

print(f'\nYOLOv7 dir  : {YOLOV7_DIR}')
print(f'Fine-tuned  : {WEIGHTS_PATH}  ({os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB)')
print('Setup OK.')

Gradio 6.14.0 already installed.

YOLOv7 dir  : /Users/jatinnabhoya/Desktop/UNH/Semester 3/Deep Learning /YOLOv7_Gym_Equipment_Detection/yolov7
Fine-tuned  : /Users/jatinnabhoya/Desktop/UNH/Semester 3/Deep Learning /YOLOv7_Gym_Equipment_Detection/best.pt  (146.1 MB)
Setup OK.


In [11]:
# ── Step 1: Imports & Config ──────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import cv2
import gradio as gr
from PIL import Image

sys.path.insert(0, YOLOV7_DIR)
from models.experimental import attempt_load
from utils.general        import non_max_suppression, scale_coords
from utils.torch_utils    import select_device
from utils.datasets       import letterbox

DEVICE      = select_device('')   # auto: CUDA > MPS > CPU
IMG_SIZE    = 640
CLASS_NAMES = ['dumbbell', 'barbell', 'kettlebell', 'resistance_band', 'pull_up_bar']
NUM_CLASSES = len(CLASS_NAMES)

# One BGR color per class (used with cv2 drawing on RGB images — values still work)
COLORS = [
    (220,  50,  50),   # dumbbell      — red
    ( 50, 180,  50),   # barbell       — green
    ( 50, 100, 220),   # kettlebell    — blue
    (240, 180,  30),   # resistance_band — gold
    (150,  60, 210),   # pull_up_bar   — purple
]

print(f'Device  : {DEVICE}')
print(f'Classes : {CLASS_NAMES}')

Device  : cpu
Classes : ['dumbbell', 'barbell', 'kettlebell', 'resistance_band', 'pull_up_bar']


In [12]:
# ── Step 2: Build Architecture & Load Fine-Tuned Weights ──────────────────────
#
# best.pt is a plain state_dict (saved after replacing the detection head).
# We must rebuild the exact same 5-class architecture before calling load_state_dict.
# This mirrors build_model() in 02_phase2_finetuning.ipynb exactly.

def build_model():
    # 1. Load YOLOv7 COCO backbone + original 80-class head
    m = attempt_load(COCO_WEIGHTS, map_location=DEVICE)
    detect = m.model[-1]        # IDetect — the output detection module
    na     = detect.na          # 3 anchors per scale
    new_no = NUM_CLASSES + 5    # 10 outputs per anchor (xywh + obj + 5 classes)

    # 2. Replace 80-class Conv2d layers with 5-class ones (same in_channels)
    for i, conv in enumerate(detect.m):
        detect.m[i] = nn.Conv2d(conv.in_channels, new_no * na, 1).to(DEVICE)

    detect.nc = NUM_CLASSES
    detect.no = new_no
    m.nc      = NUM_CLASSES
    m.names   = CLASS_NAMES
    return m

model = build_model()
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

total = sum(p.numel() for p in model.parameters())
print(f'Fine-tuned model ready — {total:,} parameters, {NUM_CLASSES} classes')
# "detect" was a local variable inside build_model(); retrieve the Detect module from the model
detect = model.model[-1]
print(f'Detection head output: {list(detect.m[0].weight.shape)} per scale')

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
Fine-tuned model ready — 36,501,466 parameters, 5 classes
Detection head output: [30, 256, 1, 1] per scale


In [13]:
# ── Step 3: Inference & Visualization Helpers ─────────────────────────────────

def detect_objects(pil_image, conf_thres=0.25, iou_thres=0.45):
    """
    Run YOLOv7 on a PIL Image.
    Returns:
        annotated  : numpy RGB array with bounding boxes drawn
        detections : list of (class_name, conf, x1, y1, x2, y2)
    """
    img_orig = np.array(pil_image.convert('RGB'))   # ensure RGB

    # Letterbox to 640×640 preserving aspect ratio (grey pad = 114)
    img_lb, _, _ = letterbox(img_orig, IMG_SIZE, auto=False, scaleFill=False)

    # HWC → CHW, float [0,1], batch dim
    img_t = (
        torch.from_numpy(img_lb).float() / 255.0
    ).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        raw = model(img_t)[0]

    # NMS — returns list of detections per image; we only have 1 image
    dets = non_max_suppression(raw, conf_thres=conf_thres, iou_thres=iou_thres)[0]

    annotated  = img_orig.copy()
    detections = []

    if dets is not None and len(dets):
        # Rescale boxes from 640×640 letterbox space → original image pixels
        dets[:, :4] = scale_coords(
            img_t.shape[2:], dets[:, :4], img_orig.shape
        ).round()

        for *xyxy, conf, cls_id in dets.cpu().numpy():
            x1, y1, x2, y2 = map(int, xyxy)
            cls_id = int(cls_id)
            color  = COLORS[cls_id]
            label  = f'{CLASS_NAMES[cls_id]} {conf:.2f}'

            # Draw bounding box
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

            # Label background + white text
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(annotated, (x1, max(0, y1 - th - 8)), (x1 + tw + 4, y1), color, -1)
            cv2.putText(
                annotated, label, (x1 + 2, max(4, y1 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA
            )

            detections.append((CLASS_NAMES[cls_id], float(conf), x1, y1, x2, y2))

    return annotated, detections


def make_side_by_side(pil_image, annotated_np):
    """Stack original and annotated images side-by-side with header banners."""
    orig = np.array(pil_image.convert('RGB'))

    def add_banner(img, title):
        """Prepend a dark banner strip with a text label."""
        banner = np.full((36, img.shape[1], 3), 30, dtype=np.uint8)
        cv2.putText(banner, title, (8, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (220, 220, 220), 2, cv2.LINE_AA)
        return np.vstack([banner, img])

    left  = add_banner(orig,         'Original')
    right = add_banner(annotated_np, 'Detected')

    # Pad shorter panel to same height
    h = max(left.shape[0], right.shape[0])
    def pad_h(x):
        diff = h - x.shape[0]
        return np.vstack([x, np.full((diff, x.shape[1], 3), 30, dtype=np.uint8)]) if diff > 0 else x
    left, right = pad_h(left), pad_h(right)

    # 6-pixel dark divider between panels
    divider = np.full((h, 6, 3), 60, dtype=np.uint8)
    combined = np.hstack([left, divider, right])
    return Image.fromarray(combined)


def gradio_predict(pil_image, conf_thres, iou_thres):
    """Main Gradio callback: image + sliders → side-by-side image + summary text."""
    if pil_image is None:
        return None, 'Upload an image to get started.'

    annotated, detections = detect_objects(pil_image, conf_thres=conf_thres, iou_thres=iou_thres)
    result_img = make_side_by_side(pil_image, annotated)

    if not detections:
        summary = f'No objects detected above confidence {conf_thres:.2f}.'
    else:
        lines = [f'{len(detections)} detection(s) found:']
        for name, conf, x1, y1, x2, y2 in detections:
            lines.append(f'  • {name}: {conf:.2f}  [box {x1},{y1} → {x2},{y2}]')
        summary = '\n'.join(lines)

    return result_img, summary


print('Inference pipeline ready. Run the next cell to launch the app.')

Inference pipeline ready. Run the next cell to launch the app.


In [14]:
# ── Collect example images from test split ────────────────────────────────────
import glob

test_imgs = sorted(
    glob.glob('../dataset/images/test/*.jpg') +
    glob.glob('../dataset/images/test/*.jpeg') +
    glob.glob('../dataset/images/test/*.png')
)
examples = [[p, 0.25, 0.45] for p in test_imgs[:4]] if test_imgs else None
print(f'{len(test_imgs)} test images found — {len(examples) if examples else 0} shown as examples.')

21 test images found — 4 shown as examples.


In [17]:
# ── Step 4: Launch Gradio App ─────────────────────────────────────────────────

demo = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Image(type='pil', label='Input Image'),
        gr.Slider(0.05, 0.90, value=0.25, step=0.05, label='Confidence Threshold'),
        gr.Slider(0.10, 0.90, value=0.45, step=0.05, label='IoU Threshold (NMS)'),
    ],
    outputs=[
        gr.Image(type='pil', label='Original  |  Detected'),
        gr.Textbox(label='Detection Summary', lines=8),
    ],
    title='YOLOv7 Gym Equipment Detector',
    description=(
        'Upload a gym image to detect: **dumbbell · barbell · kettlebell · '
        'resistance_band · pull_up_bar**\n'
        'Fine-tuned YOLOv7 on 203 custom images — mAP@0.5 = 0.46 (test set + TTA)\n'
        'UNH Deep Learning, Spring 2026 — varun · Mohit · Jatin'
    ),
    examples=examples,
    flagging_mode='never',   # Gradio 6.0: replaces allow_flagging
)

# share=False → local only at http://127.0.0.1:7860
# share=True  → also generates a public gradio.live URL (useful for recording)
# theme moved from Interface() to launch() in Gradio 6.0
demo.launch(share=True, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://a2dbfc992f2751043a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Recording the Demo Video

1. **Run all cells above** — Gradio opens at `http://127.0.0.1:7860`
2. **Start screen recording**
   - macOS: `⌘ Shift 5` → Select Window or Screen → Record
   - Or: QuickTime Player → File → New Screen Recording
3. **In the Gradio app, show:**
   - Upload 2–3 images from `dataset/images/test/`
   - Confirm the model detects the correct equipment class
   - Drag the Confidence slider — show how it filters detections
   - Point out the side-by-side layout (original left, annotated right)
4. **Stop recording** → export as `.mp4`

> **Tip:** Set `share=True` in `demo.launch()` to get a public `gradio.live` link — useful if you want the URL on screen or need to demo from another device.

**Submit:** `notebooks/04_gradio_deployment.ipynb` + the `.mp4` video file.